<img src="http://developer.download.nvidia.com/notebooks/dlsw-notebooks/rivaasrasr-finetuning-conformer-ctc-nemo/nvidia_logo.png" style="width: 90px; float: right;">

# Fine-Tuning Nemotron 3.5 ASR (Streaming) for an Out-of-Set Language: Frisian

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jellewas/frisian-asr/blob/main/asr-finetune-nemotron-3.5-asr-streaming-frisian.ipynb)

This tutorial fine-tunes [`nvidia/nemotron-3.5-asr-streaming-0.6b`](https://huggingface.co/nvidia/nemotron-3.5-asr-streaming-0.6b) — a 600M-parameter Cache-Aware FastConformer-RNNT streaming model — onto **West Frisian (`fy-NL`)**, a language that is **not** one of the model's 40 supported locales.

It is a companion to the official [Nemotron 3.5 ASR streaming-prompt fine-tune notebook](https://github.com/nvidia-riva/tutorials/blob/main/asr-finetune-nemotron-3.5-asr-streaming-prompt.ipynb) and the blog [*How to Fine-Tune Nemotron 3.5 ASR for Your Language, Domain, or Accent*](https://huggingface.co/blog/nvidia/fine-tuning-nemotron-35-asr). Those examples fine-tune locales that **already exist** in the 40 (Greek, Bulgarian). This one shows the harder case: **a language with no prompt slot at all**, and is faithful to NVIDIA's recipe on every load-bearing axis. Every deviation is flagged in-place.

> **Community contribution, not an NVIDIA model.** This notebook and the checkpoint it produces are a community derivative of NVIDIA's Nemotron 3.5 ASR. NVIDIA did not produce, endorse, or review them. "Nemotron" is a trademark of NVIDIA, used here only to identify the base model.

## What you will build

Frisian is not in the 40 locales, so the base model cannot transcribe it — under any language prompt it produces garbled, Dutch-like text (~83% WER). A single full fine-tune on ~40 h of Frisian, conditioned on the **closest supported locale slot (`nl-NL`, Dutch)**, takes it to **~20% raw WER at 80 ms streaming** — a **75% relative reduction** — on a leak-free, speaker- and sentence-disjoint test set.

| Latency (`att_context_size`) | Base WER | Fine-tuned WER | Rel. improvement |
|---|---|---|---|
| 80 ms `[56,0]`   | 82.87% | **20.36%** | **75.4%** |
| 1120 ms `[56,13]`| 81.69% | **18.01%** | **77.9%** |

*Raw WER (NeMo `word_error_rate`, no normalization), cache-aware streaming, 3,173-clip held-out test, same evaluation for base and fine-tuned.*

## Steps
1. Prerequisites & environment
2. Load the published `LokaalHub/frisian-asr-cv22` dataset and write NeMo dual-tag manifests
3. Restore the base checkpoint
4. Fine-tune with the NVIDIA streaming-prompt recipe
5. Cache-aware streaming evaluation (NVIDIA-format raw-WER tables)
6. Results, deviations, and references

## 1. Prerequisites & environment

**Hardware — needs a CUDA GPU.** NeMo RNNT training and cache-aware streaming inference do **not** run on Apple Silicon (MPS) or CPU, so this notebook will **not** run natively on a Mac. We used **1× A100 80GB**; ~24 GB is enough for the documented `batch_duration`.

**On a Mac, or any machine without an NVIDIA GPU → use Colab.** Click the **Open in Colab** badge at the top, then **Runtime → Change runtime type → GPU**. On a free 16 GB T4, lower `model.train_ds.batch_duration` (e.g. to `~100`) if you hit out-of-memory; Colab Pro (L4 / A100) runs the documented settings as-is.

**Software.** An NVIDIA NeMo container (e.g. `nvcr.io/nvidia/nemo:25.07` or newer) ships NeMo preinstalled; otherwise the install cell below adds it. The prompt-conditioned config currently ships **only on NeMo `main`** — no tagged release includes it yet — so we install from `main`, matching NVIDIA's official notebook (which pins `BRANCH = 'main'`, *“subject to change to a fixed tag / version upon release”*).

NVIDIA's prerequisites for fine-tuning this model (from the [notebook](https://github.com/nvidia-riva/tutorials/blob/main/asr-finetune-nemotron-3.5-asr-streaming-prompt.ipynb) and [blog](https://huggingface.co/blog/nvidia/fine-tuning-nemotron-35-asr)), and how this tutorial meets them:

| NVIDIA prerequisite | How we satisfy it |
|---|---|
| 16 kHz **mono** audio | Dataset is stored at 16 kHz; manifest builder resamples to mono 16 kHz if needed |
| NeMo JSON-lines manifest (`audio_filepath`, `duration`, `text`) | Written below, plus the dual `lang`/`target_lang` tags |
| `target_lang` prompt tag on every clip | Every clip tagged `target_lang=nl-NL` (see Step 2 for the out-of-set rationale) |
| Punctuated, properly-cased text (match base style) | We keep CV casing/punctuation; **raw** WER reported (stricter) |
| Reuse pretrained tokenizer when data < 50 h | ~40 h → tokenizer reused unchanged (no vocab surgery) |
| Fixed **step budget** for streaming/iterable data | `trainer.max_steps=8000` (not epochs) |
| Evaluate at deployment latency on held-out data | Cache-aware streaming eval at `[56,0]` (0 ms look-ahead) on the disjoint test split |

The install block below mirrors the official notebook. For an exact, reproducible build, replace `main` with a specific commit SHA once a tagged release ships the prompt config.

In [ ]:
# Skip this cell if you are running inside an NGC NeMo container (>= 25.07), where NeMo is preinstalled.
# The prompt-conditioned config ships only on NeMo `main` today (no tagged release yet),
# so we track `main`, matching NVIDIA's official notebook. Pin a commit SHA for exact reproducibility.
BRANCH = "main"  # subject to change to a fixed tag / version upon release

import os
if not os.environ.get("NEMO_PREINSTALLED"):
    !apt-get -qq install -y sox libsndfile1 ffmpeg libsox-fmt-mp3 >/dev/null
    !pip install -q text-unidecode "matplotlib>=3.3.2" Cython packaging
    !pip install -q "nemo_toolkit[asr] @ git+https://github.com/NVIDIA-NeMo/NeMo.git@{BRANCH}"

# datasets + jiwer for data loading and the offline sanity-WER; soundfile/librosa for resampling.
!pip install -q "datasets>=2.18" jiwer soundfile librosa huggingface_hub

In [ ]:
# Clone NeMo for the example scripts (speech_to_text_finetune.py, the prompt config, and the
# cache-aware streaming inference script). Match the release you installed above.
import os
if not os.path.isdir("NeMo"):
    !git clone -b {BRANCH} --depth 1 https://github.com/NVIDIA-NeMo/NeMo.git
NEMO_DIR = os.path.abspath("NeMo")
print("NeMo scripts at:", NEMO_DIR)

## 2. Data: load the published dataset and write dual-tag manifests

We start from the already-published, leak-free dataset [`LokaalHub/frisian-asr-cv22`](https://huggingface.co/datasets/LokaalHub/frisian-asr-cv22) (CC0, 49.6 h) so data-prep stays light. It was derived from **Common Voice 22.0** `fy-NL`, with **speaker- and sentence-disjoint** splits and a per-clip `quality_score`.

| Split | Hours | Notes |
|---|---|---|
| train | 40.4 h | 3,924 validated clips + 26,005 quality-filtered Common Voice `other` clips |
| dev   | 4.6 h  | validation during training |
| test  | 4.7 h  | 3,173 clips, held out |

<details><summary><b>How the dataset itself was built (summary — not re-run here)</b></summary>

Common Voice 22.0 `fy-NL` was loaded; `test`/`dev` are CV's high-confidence splits; `train` = validated remainder **plus** a CER-filtered slice of the unvalidated `other` bucket. Each `other` clip was CTC-decoded with a Frisian wav2vec2 model (`greenw0lf/wav2vec2-large-xls-r-1b-frisian`) and kept only if CER vs. the prompt ≤ 0.25. Train is disjoint from eval by **speaker** (`client_id`) **and** sentence. Audio is 16 kHz mono; clips clamped to 1–20 s. Full build script: [`hf_job.py`](https://github.com/jellewas/frisian-asr/blob/main/hf_job.py) in the source repo.

</details>

### Language strategy — a deliberate, flagged deviation

> **⚠️ Deviation 1 — riding the `nl-NL` prompt slot for an out-of-set language.** Nemotron 3.5 ASR's language prompt is a fixed embedding over its 40 locales. Frisian is not one of them, and an invented `<fy-NL>` tag would map to an **untrained** slot. NVIDIA's own guidance is emphatic: *"Get the language tag right — the prompt conditioning is powerful but unforgiving of mismatched language labels."* Their worked examples (Greek, Bulgarian) fine-tune locales that **already exist**. Here, no slot exists, so we tag every clip with the **nearest** high-resource locale — `nl-NL` (Dutch), Frisian's closest orthographic/phonetic neighbour — to get a warm start, then specialize that slot to Frisian. This is an **extension of**, not part of, NVIDIA's documented procedure. After fine-tuning, `nl-NL` no longer means "Dutch" for this checkpoint; it means "Frisian."

**Manifest format.** NVIDIA's manifest line is `{"audio_filepath", "duration", "text", "lang", "target_lang"}`. The training config reads the prompt from `target_lang`; the inference scripts read `lang`. We write **both** with the same value so one set of manifests serves training, validation, and eval.

In [ ]:
import io, json, os
import soundfile as sf
import librosa
from datasets import Audio, load_dataset

SR = 16000                                   # NVIDIA prerequisite: 16 kHz mono
DATASET = "LokaalHub/frisian-asr-cv22"
TARGET_LANG = "nl-NL"                         # Frisian rides the nearest trained slot (Deviation 1)
HF_TOKEN = os.environ.get("HF_TOKEN")        # optional: this dataset is public/ungated (CC0), so None works


def _decode(a):
    src = io.BytesIO(a["bytes"]) if a.get("bytes") else a["path"]
    arr, sr = sf.read(src, dtype="float32")
    if arr.ndim > 1:                         # force mono
        arr = arr.mean(axis=1)
    return arr, sr


def build_manifest(split, path, limit=None):
    """Materialize 16 kHz mono WAVs + a NeMo dual-tag manifest for one split."""
    ds = load_dataset(DATASET, split=split, token=HF_TOKEN)
    ds = ds.cast_column("audio", Audio(decode=False))
    if limit:
        ds = ds.select(range(min(limit, len(ds))))
    wav_dir = os.path.abspath(f"wav/{split}")
    os.makedirs(wav_dir, exist_ok=True)
    with open(path, "w") as f:
        for i, ex in enumerate(ds):
            arr, sr = _decode(ex["audio"])
            if sr != SR:
                arr = librosa.resample(arr, orig_sr=sr, target_sr=SR)
            wav = os.path.join(wav_dir, f"{i:05d}.wav")
            sf.write(wav, arr, SR)
            f.write(json.dumps({
                "audio_filepath": wav,
                "duration": round(len(arr) / SR, 3),
                "text": ex["text"],            # keep CV casing + punctuation (match base style)
                "lang": TARGET_LANG,           # read by the inference scripts
                "target_lang": TARGET_LANG,    # read by the training config
            }, ensure_ascii=False) + "\n")
    print(f"{split}: {sum(1 for _ in open(path))} clips -> {path}")


# Full run (slow: writes ~33k WAVs). For a wiring smoke test, pass e.g. limit=30 to each call.
build_manifest("train", "train.json")
build_manifest("dev",   "dev.json")
build_manifest("test",  "test.json")

In [ ]:
# Sanity-check one manifest line against NVIDIA's format.
print(open("test.json").readline().strip())
# Expected (one example):
# {"audio_filepath": ".../wav/test/00000.wav", "duration": 4.27, "text": "In moaie dei.",
#  "lang": "nl-NL", "target_lang": "nl-NL"}

## 3. Restore the base checkpoint

Download the base model and save it as a local `.nemo` so we can pass it to `init_from_nemo_model`. Architecture (from the [model card](https://huggingface.co/nvidia/nemotron-3.5-asr-streaming-0.6b)): Cache-Aware FastConformer-RNNT, **600M params, 24 encoder layers, `d_model=512`**, prompt conditioning over 40 locales.

In [ ]:
import nemo.collections.asr as nemo_asr

BASE_MODEL = "nvidia/nemotron-3.5-asr-streaming-0.6b"
base = nemo_asr.models.ASRModel.from_pretrained(model_name=BASE_MODEL)
base.save_to("base.nemo")
del base
print("saved base.nemo")

## 4. Fine-tune (NVIDIA streaming-prompt recipe)

Full fine-tune with NeMo's `speech_to_text_finetune.py` and the prompt-conditioned cache-aware streaming config `fastconformer_transducer_bpe_streaming_prompt`, initialized from the base checkpoint via `init_from_nemo_model`. This is the **same script and config** as the official notebook.

**Recipe (faithful to NVIDIA):** AdamW, base LR `0.1` with **NoamAnnealing**, `weight_decay=0.001`, `batch_duration=200` (dynamic batching by audio seconds), `bf16`, dual-tag manifests, tokenizer reused from base (< 50 h).

> **⚠️ Deviation 2 — `sched.d_model=512` (notebook uses `1024`).** The NoamAnnealing peak LR scales as `lr · d_model^-0.5 · warmup^-0.5`, so `d_model` must match the encoder. The official notebook hard-codes `1024` (a generic template value), but this 0.6B checkpoint's true encoder dim is **512** (per its model card). `512` is the architecturally-correct constant here. With `lr=0.1`, `warmup=500`: effective peak ≈ `0.1 · 512^-0.5 · 500^-0.5 ≈ 2.0e-4`.
>
> **⚠️ Deviation 3 — `warmup_steps=500` (notebook: `100`) and `max_steps=8000` (notebook demo: 1 epoch / 60 batches).** The notebook's tiny `limit_train_batches=60`, 1-epoch settings are a *demo*, not a production schedule. We use a gentler warmup and a real **step budget**, which NVIDIA's blog calls *"the right way to schedule with streaming/iterable data."* These are benign free hyperparameters.

> **⚠️ Deviation 4 — monolingual, no replay.** NVIDIA recommends *"blend in a slice of the model's other languages ('replay') ... so you sharpen your target locales without eroding the rest."* We **intentionally omit replay** to produce a dedicated Frisian model. The cost is documented and real: this checkpoint **catastrophically forgets the other 39 locales** and transcribes (almost) any input as Frisian. **If you want a multilingual result, add replay data here** — mix a slice of the base model's other locales into `train.json` (each with its own correct `target_lang`) and re-check those locales after training. We skip it on purpose; you probably should not.

In [ ]:
# Fine-tune. On 1x A100 80GB this is ~1h35m wall-clock for max_steps=8000.
# Heavy GPU cell -- expected best validation WER (greedy, dev) ~0.233.
CFG_DIR  = f"{NEMO_DIR}/examples/asr/conf/fastconformer/cache_aware_streaming"
CFG_NAME = "fastconformer_transducer_bpe_streaming_prompt"

!python {NEMO_DIR}/examples/asr/speech_to_text_finetune.py \
    --config-path={CFG_DIR} \
    --config-name={CFG_NAME} \
    +init_from_nemo_model=$(pwd)/base.nemo \
    model.train_ds.manifest_filepath=$(pwd)/train.json \
    model.validation_ds.manifest_filepath=$(pwd)/dev.json \
    model.train_ds.is_tarred=false \
    model.train_ds.batch_duration=200 \
    model.optim.name=adamw \
    model.optim.lr=0.1 \
    model.optim.weight_decay=0.001 \
    model.optim.sched.warmup_steps=500 \
    model.optim.sched.d_model=512 \
    trainer.max_steps=8000 \
    trainer.devices=1 \
    trainer.precision=bf16 \
    trainer.accelerator=gpu \
    exp_manager.exp_dir=$(pwd)/exp \
    exp_manager.name=frisian_ft

In [ ]:
# Locate the trained checkpoint produced by exp_manager.
import glob, os
cands = sorted(glob.glob("exp/**/*.nemo", recursive=True), key=os.path.getmtime)
assert cands, "no trained .nemo produced -- inspect the training logs above"
FT_MODEL = os.path.abspath(cands[-1])
print("fine-tuned checkpoint:", FT_MODEL)

## 5. Cache-aware streaming evaluation

We evaluate with NVIDIA's own [`speech_to_text_cache_aware_streaming_infer.py`](https://github.com/NVIDIA/NeMo/blob/main/examples/asr/asr_cache_aware_streaming/speech_to_text_cache_aware_streaming_infer.py) so our numbers sit next to NVIDIA's without an asterisk. The metric the script logs (`WER% of streaming mode`) is **raw WER** (NeMo `word_error_rate`, no normalization) — NVIDIA's reporting convention, and stricter than a normalized WER.

Key flags (matching NVIDIA): `decoder_type=rnnt`, `target_lang=nl-NL`, `strip_lang_tags=true` (removes the `<nl-NL>` tag before scoring), `pad_and_drop_preencoded=true`. We sweep the full **att_context_size latency ladder**:

| `att_context_size` | Latency | Use case |
|---|---|---|
| `[56,0]`  | 80 ms   | ultra-low-latency voice agents (NVIDIA's headline) |
| `[56,1]`  | 160 ms  | interactive voice agents |
| `[56,3]`  | 320 ms  | conversational AI / live caption |
| `[56,6]`  | 560 ms  | higher accuracy |
| `[56,13]` | 1120 ms | highest accuracy |

The **same checkpoint** covers the whole ladder — you pick the operating point at inference, no retraining.

In [ ]:
import os, re, subprocess

INFER = f"{NEMO_DIR}/examples/asr/asr_cache_aware_streaming/speech_to_text_cache_aware_streaming_infer.py"
LAT = {"[56,0]": "80 ms", "[56,1]": "160 ms", "[56,3]": "320 ms",
       "[56,6]": "560 ms", "[56,13]": "1120 ms"}


def stream_wer(model_path, att_context):
    """Run NVIDIA's cache-aware streaming eval and return the raw WER% it logs."""
    out = os.path.abspath("stream_out_" + att_context.strip("[]").replace(",", "_"))
    cmd = (f"python {INFER} model_path={model_path} "
           f"dataset_manifest={os.path.abspath('test.json')} output_path={out} "
           f"target_lang={TARGET_LANG} att_context_size=\"{att_context}\" "
           f"decoder_type=rnnt pad_and_drop_preencoded=true "
           f"strip_lang_tags=true batch_size=32")
    res = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    print(res.stdout[-1500:])
    m = re.findall(r"WER% of streaming mode:\s*([0-9.]+)", res.stdout)
    assert m, f"could not parse WER for {att_context} -- see log above"
    wer = float(m[-1])
    print(f"att={att_context} ({LAT[att_context]}) RAW WER = {wer:.2f}%")
    return wer

In [ ]:
# Base model at the two anchor latencies (the 'before').
# Heavy GPU cell. Expected output shown below so this need not be re-run.
base_wer = {att: stream_wer(os.path.abspath("base.nemo"), att) for att in ["[56,0]", "[56,13]"]}
print(base_wer)
# Expected: {'[56,0]': 82.87, '[56,13]': 81.69}

In [ ]:
# Fine-tuned model across the full ladder (the 'after').
# Heavy GPU cell. Expected output shown below.
ft_wer = {att: stream_wer(FT_MODEL, att) for att in ["[56,0]", "[56,1]", "[56,3]", "[56,6]", "[56,13]"]}
print(ft_wer)
# Expected: {'[56,0]': 20.36, '[56,1]': 19.78, '[56,3]': 18.70, '[56,6]': 18.40, '[56,13]': 18.01}

In [ ]:
# Render the NVIDIA-format result tables.
b0, f0 = base_wer["[56,0]"], ft_wer["[56,0]"]
rel = (b0 - f0) / b0 * 100
print("Raw WER (%), held-out Frisian test, cache-aware streaming, lowest-latency [56,0] (80 ms):\n")
print("| Language        | Base Model WER | Fine-tuned WER | Relative Improvement |")
print("|-----------------|----------------|----------------|----------------------|")
print(f"| Frisian (nl-NL) | {b0:13.0f}% | {f0:13.0f}% | {rel:19.0f}% |\n")
print("Fine-tuned latency ladder (raw WER %):")
print("| att_context_size | Latency | WER   |")
print("|------------------|---------|-------|")
for att in ["[56,0]", "[56,1]", "[56,3]", "[56,6]", "[56,13]"]:
    print(f"| {att:>16} | {LAT[att]:>7} | {ft_wer[att]:4.1f}% |")

## 6. Results

Expected output of the cell above (our published run on the 3,173-clip held-out test):

```
Raw WER (%), held-out Frisian test, cache-aware streaming, lowest-latency [56,0] (80 ms):

| Language        | Base Model WER | Fine-tuned WER | Relative Improvement |
|-----------------|----------------|----------------|----------------------|
| Frisian (nl-NL) |            83% |            20% |                  75% |

Fine-tuned latency ladder (raw WER %):
| att_context_size | Latency | WER   |
|------------------|---------|-------|
|           [56,0] |   80 ms | 20.4% |
|           [56,1] |  160 ms | 19.8% |
|           [56,3] |  320 ms | 18.7% |
|           [56,6] |  560 ms | 18.4% |
|          [56,13] | 1120 ms | 18.0% |
```

**Takeaways**
- Frisian goes from untranscribable (~83% WER) to **20.4% raw WER at 80 ms** streaming — a **75% relative reduction** — with one full fine-tune on ~40 h, no replay, tokenizer reused.
- The ~2.3-point spread across the ladder (20.4% @ 80 ms → 18.0% @ 1120 ms) matches the ~9–10% relative streaming penalty NVIDIA documents at the lowest-latency setting.
- This mirrors NVIDIA's Greek/Bulgarian result *shape* (biggest wins where the base was weakest) — here taken to the extreme of a language with **no prompt slot at all**.

### Using the fine-tuned model
Frisian rides the `nl-NL` slot, so inference **must** pass `target_lang=nl-NL`:
```bash
python NeMo/examples/asr/asr_cache_aware_streaming/speech_to_text_cache_aware_streaming_infer.py \
    model_path=frisian.nemo dataset_manifest=clips.json \
    target_lang=nl-NL att_context_size="[56,0]" decoder_type=rnnt strip_lang_tags=true
```

### Summary of deviations from NVIDIA's recipe
| # | Deviation | Justification |
|---|---|---|
| 1 | Tag out-of-set Frisian as `nl-NL` | No `fy` slot exists; `nl-NL` is the nearest trained locale (warm start) |
| 2 | `sched.d_model=512` (vs 1024) | Matches this checkpoint's true encoder dim; corrects the NoamAnnealing constant |
| 3 | `warmup_steps=500`, `max_steps=8000` | Real step budget vs the notebook's 60-batch demo; benign free params |
| 4 | No replay (monolingual) | Deliberate dedicated-Frisian goal; **forgets other locales** — add replay for a multilingual result |

### References
- Notebook: [Nemotron 3.5 ASR streaming-prompt fine-tune (nvidia-riva/tutorials)](https://github.com/nvidia-riva/tutorials/blob/main/asr-finetune-nemotron-3.5-asr-streaming-prompt.ipynb)
- Blog: [How to Fine-Tune Nemotron 3.5 ASR for Your Language, Domain, or Accent](https://huggingface.co/blog/nvidia/fine-tuning-nemotron-35-asr)
- Base model: [nvidia/nemotron-3.5-asr-streaming-0.6b](https://huggingface.co/nvidia/nemotron-3.5-asr-streaming-0.6b)
- Framework: [NVIDIA NeMo](https://github.com/NVIDIA/NeMo)
- Dataset (CC0): [LokaalHub/frisian-asr-cv22](https://huggingface.co/datasets/LokaalHub/frisian-asr-cv22)
- Community fine-tune produced by this recipe: [LokaalHub/frisian-asr-streaming-0.6b](https://huggingface.co/LokaalHub/frisian-asr-streaming-0.6b)